In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import current_timestamp, current_date, input_file_name, lit
import datetime

STREAM_NAME   = "tariff_metrics"
BUCKET        = "s3://energy-pipline-prod"
RAW_PATH      = f"{BUCKET}/raw/tariff_metrics_stream_v2.csv"
BRONZE_PATH   = f"{BUCKET}/bronze/tariff_metrics/"         # ← updated
CATALOG_TABLE = "energy_catalog.raw.bronze_tariff_metrics"
BAD_RECORDS   = f"{BUCKET}/bronze/logs/bad_records/tariff_metrics/"  # ← updated
batch_date    = datetime.date.today().strftime("%Y-%m-%d")

print(f"Source : {RAW_PATH}")
print(f"Target : {BRONZE_PATH}")
print(f"Date   : {batch_date}")

In [0]:
tariff_schema = StructType([
    StructField("household_id",      StringType(),  True),
    StructField("tariff_region",     StringType(),  True),
    StructField("tariff_city",       StringType(),  True),
    StructField("tariff_plan_type",  StringType(),  True),
    StructField("billing_cycle",     StringType(),  True),
    StructField("utility_provider",  StringType(),  True),
    StructField("unit_rate",         DoubleType(),  True),
    StructField("peak_rate",         DoubleType(),  True),
    StructField("offpeak_rate",      DoubleType(),  True),
    StructField("fixed_charge",      DoubleType(),  True),
    StructField("tax_amount",        DoubleType(),  True),
    StructField("subsidy_amount",    DoubleType(),  True),
    StructField("monthly_bill",      DoubleType(),  True),
    StructField("billing_units",     DoubleType(),  True),
    StructField("late_fee",          DoubleType(),  True),
    StructField("adjustment_amount", DoubleType(),  True),
])
print(f"Schema ready: {len(tariff_schema.fields)} columns")

In [0]:
df_raw = spark.table("energy_catalog.raw.src_tariff_metrics")
print(f"Rows read: {df_raw.count()}")   
print(f"Columns: {len(df_raw.columns)}")  
df_raw.show(3)

In [0]:
from pyspark.sql.functions import col

df_bronze = (
    df_raw
    .withColumn("_batch_date",   lit(batch_date))
    .withColumn("_ingestion_ts", current_timestamp())
    .withColumn("_source_file",  col("_metadata.file_path"))  # ← fixed
    .withColumn("_stream_name",  lit(STREAM_NAME))
)
print(f"Total columns: {len(df_bronze.columns)}")  # expect 21

In [0]:
(df_bronze.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("_batch_date")
    .save(BRONZE_PATH)
)
print("Write complete")
print(f"Rows in Delta: {spark.read.format('delta').load(BRONZE_PATH).count()}")

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {CATALOG_TABLE}
    USING DELTA
    LOCATION '{BRONZE_PATH}'
""")
spark.sql(f"SELECT COUNT(*) AS total FROM {CATALOG_TABLE}").show()

In [0]:
spark.sql(f"OPTIMIZE {CATALOG_TABLE} ZORDER BY (household_id)")
print("Done. Bronze_tariff_metrics complete.") 